In [1]:
%load_ext autoreload
%autoreload 2

import os
import re
import pandas as pd
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import your scraper function from the .py file
from scraper_utils import run_scraper

# Define parameters dynamically at runtime
path = os.getenv("PORTAL")
BRONZE = os.getenv("BRONZENEW")

states_input = {
    "22": "CHHATTISGARH"
}

# --- DYNAMIC ACTIVITIES_INPUT GENERATION & STATE ITERATION START ---
stact_path = os.getenv("STACT")
if not stact_path:
    raise ValueError("⚠️ STACT environment variable is not set.")

if not stact_path.lower().endswith((".xlsx", ".xls")):
    stact_path += ".xlsx"

if not os.path.exists(stact_path):
    raise FileNotFoundError(f"⚠️ STACT file not found at: {stact_path}")

# Read the STACT Excel file once
stact_df = pd.read_excel(stact_path)

# Standardize Location values in Excel for comparison
stact_df["_location_clean"] = (
    stact_df["Location"].astype(str).str.strip().str.upper()
)

# Regex to extract key and value from strings like: "15": "4(c) Asbestos milling...",
mapping_pattern = re.compile(r'^\s*"(.*?)":\s*"(.*?)"\s*,\s*$')

# Iterate over each state individually to optimize execution per state
for state_code, state_name in states_input.items():
    current_state_input = {state_code: state_name}
    target_state_clean = state_name.strip().upper()
    
    print(f"\n==================================================")
    print(f"🔄 Processing State: {state_name} (Code: {state_code})")
    print(f"==================================================")

    # Filter rows for the current state only
    matching_rows = stact_df[
        stact_df["_location_clean"] == target_state_clean
    ]

    # DEFENSIVE CHECK 1: Identify if State is Missing in STACT Excel
    if matching_rows.empty:
        print(f"⚠️ WARNING: State '{state_name}' from states_input was NOT found in the STACT Excel file. Skipping...")
        continue

    activities_input = {}
    unique_mappings = matching_rows["Activity Mapping"].dropna().unique()

    for entry in unique_mappings:
        entry_str = str(entry).strip()
        match = mapping_pattern.match(entry_str)
        if match:
            act_id, act_desc = match.groups()
            
            clean_act_id = act_id.strip()
            clean_act_desc = re.sub(r"\s+", " ", act_desc).strip()

            if clean_act_id not in activities_input:
                activities_input[clean_act_id] = clean_act_desc

    # DEFENSIVE CHECK 2: Ensure At Least One Activity Was Found
    if not activities_input:
        print(f"⚠️ WARNING: No valid activities found in STACT Excel for state '{state_name}'. Skipping...")
        continue

    print(f"✅ Loaded {len(activities_input)} unique activity/activities for '{state_name}':")
    for k, v in activities_input.items():
        print(f'   "{k}": "{v}"')

    # Run execution specifically for this state and its corresponding activities
    run_scraper(
        portal_url=path,
        output_base_dir=BRONZE,
        states=current_state_input,
        activities=activities_input,
        year="2026",
        headless=False  # Set to True if you don't want the browser window to open
    )
# --- DYNAMIC ACTIVITIES_INPUT GENERATION & STATE ITERATION END ---


🔄 Processing State: CHHATTISGARH (Code: 22)
✅ Loaded 13 unique activity/activities for 'CHHATTISGARH':
   "3": "1(b) Off-shore and onshore oil and gas exploration, development and production"
   "6": "1(d) Thermal Power Plants"
   "9": "2(b) Mineral beneficiation"
   "79": "2(c) Pellet Plant"
   "10": "3(a) Metallurgical Industries (ferrous and non ferrous)"
   "11": "3(b) Cement plants"
   "14": "4(b)(ii) Coaltar processing units"
   "19": "5(a) Chemical fertilizers"
   "21": "5(c) Petro-chemical complexes (industries based on processing of petroleum fractions"
   "23": "5(e ) Petroleum products and petrochemical based processing such as production of carbon black and electrode grade graphite (processes other than cracking"
   "24": "5(f) Synthetic organic chemicals industry"
   "25": "5(g) Distilleries"
   "75": "5(ga) Grain based distilleries"

🌍 Processing State: CHHATTISGARH (Value: 22)...
  └── ⚙️ Searching Activity ID: 3 (1(b) Off-shore and onshore oil...)
  ℹ️ No results found